In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the dataset
file_path_pjm = "Wagon_Summary.xlsx"
df_pjm = pd.read_excel(file_path_pjm)
file_path_pressure = "Wagon_Summary_PressureData.xlsx"
df_p = pd.read_excel(file_path_pressure)

In [ ]:
print(df_pjm.shape)
df_pjm.head()

# **Check pjm dataset quality**

*First we check from the number of InvalidDateFlag which means how many rows that we have from each folder/kit/wagon that has problem with the datetime parsing. This might mean there is file reading error (unlikely), or message incomplete. The bar chart plot percentage of invalid times within each folder, while the pie chart show the percentage of invalid with respect to total number of data*

In [ ]:
unique_values = df_pjm['Folder'].unique()
print(unique_values)

In [ ]:
# pip install folium  (once)
import folium

for folder, dfol in df_pjm.groupby('Folder'):
    if dfol.empty: 
        continue
    # center on median of all points for the folder
    lat0 = pd.concat([dfol['StartLat'], dfol['EndLat']]).median()
    lon0 = pd.concat([dfol['StartLon'], dfol['EndLon']]).median()
    fmap = folium.Map(location=[lat0, lon0], zoom_start=10, tiles='OpenStreetMap')

    for _, r in dfol.iterrows():
        if not np.isfinite([r.StartLat, r.StartLon, r.EndLat, r.EndLon]).all():
            continue
        # start marker
        folium.CircleMarker([r.StartLat, r.StartLon], radius=3, color='blue', fill=True, fill_opacity=0.6).add_to(fmap)
        # end marker
        folium.CircleMarker([r.EndLat, r.EndLon], radius=3, color='red', fill=True, fill_opacity=0.9).add_to(fmap)
        # polyline
        folium.PolyLine([[r.StartLat, r.StartLon], [r.EndLat, r.EndLon]],
                        color='orange', weight=2, opacity=0.8).add_to(fmap)

    # save one HTML per folder
    out_html = f'map_{folder}.html'
    fmap.save(out_html)
    print(f"Saved {out_html}")


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
import contextily as cx
import matplotlib.pyplot as plt

# Start from your valid dataframe
df =df_pjm.copy()

# Keep only rows with all coordinates present
mask = df[['StartLon','StartLat','EndLon','EndLat']].notna().all(axis=1)
df = df.loc[mask].copy()

# Ensure datetime
df['StartTime'] = pd.to_datetime(df['StartTime'], errors='coerce')

# Build LineString geometries (lon, lat order)
df['geometry'] = [
    LineString([(lon1, lat1), (lon2, lat2)])
    for lon1, lat1, lon2, lat2 in zip(
        df['StartLon'], df['StartLat'], df['EndLon'], df['EndLat']
    )
]

# Create GeoDataFrame with WGS84 CRS
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# Convert to Web Mercator for basemap
gdf_3857 = gdf.to_crs(epsg=3857)

# Create start points separately for annotation
gdf_3857['start_point'] = gdf_3857['geometry'].apply(lambda geom: Point(geom.coords[0]))

for folder, g in gdf_3857.groupby('Folder'):
    fig, ax = plt.subplots(figsize=(9,8))
    
    # Plot the lines
    g.plot(ax=ax, linewidth=1.5, alpha=0.8, color='orange')
    
    # Plot start points
    start_points = gpd.GeoDataFrame(geometry=g['start_point'], crs=g.crs)
    start_points.plot(ax=ax, color='green', markersize=50, zorder=3, label='Start')
    
    # Annotate with StartTime dates
    for idx, row in g.iterrows():
        start_geom = row['start_point']
        date_str = row['StartTime'].strftime('%Y-%m-%d') if pd.notna(row['StartTime']) else ''
        ax.text(start_geom.x, start_geom.y, date_str, fontsize=8, color='black', zorder=4)
    
    # Add basemap
    cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, crs=g.crs, attribution=False)
    
    # Zoom to folder data
    xmin, ymin, xmax, ymax = g.total_bounds
    pad_x = (xmax - xmin) * 0.10
    pad_y = (ymax - ymin) * 0.10
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)
    
    ax.set_axis_off()
    ax.set_title(f"Tracks over Europe – Folder: {folder}", fontsize=14)
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()


*By grouping the data using the IsMoving variable we can obtain information from each folder/kit/wagon how many percentage and when is the wagon actually traveling, so that in later step we can obtain the training data precisely from a specific wagon*

In [ ]:
# --- 1. Percentage of True/False per folder ---
folder_movement = df_pjm.groupby('Folder')['IsMoving'].value_counts(normalize=True).unstack(fill_value=0) * 100

# --- Bar chart ---
folder_movement.plot(kind='bar', stacked=True, figsize=(10,6))
plt.title('Percentage of Movement (IsMoving) by Folder')
plt.ylabel('Percentage (%)')
plt.xlabel('Folder')
plt.legend(title='IsMoving')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- 2. Overall pie chart ---
movement_counts = df_pjm['IsMoving'].value_counts()

plt.figure(figsize=(6,6))
wedges, texts, autotexts = plt.pie(
    movement_counts,
    autopct='%1.1f%%',
    startangle=90,
    colors=['skyblue', 'tomato'],
    labels=['Not Moving (False)', 'Moving (True)'],
    textprops={'fontsize':10}
)

# Add legend with counts
plt.legend(
    wedges,
    [f"{label} ({count})" for label, count in zip(['False', 'True'], movement_counts)],
    title="IsMoving",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
    fontsize=9
)

plt.title('Overall IsMoving Distribution', fontsize=14)
plt.tight_layout()
plt.show()


**Now we know that the wagon that actually has moving/travel section is Dati01, Dati05, Dati07, Dati08, Dati09, we want to label each problem that is encountered using rule based classification method.**

In [ ]:
# --- 1. Percentage and counts per folder ---
# Counts
folder_counts = df_pjm.groupby(['Folder', 'IsMoving']).size().unstack(fill_value=0)

# Percentages
folder_percent = folder_counts.div(folder_counts.sum(axis=1), axis=0) * 100

# Combine into one DataFrame
folder_stats = folder_percent.round(1).astype(str) + "% (" + folder_counts.astype(str) + ")"
print(folder_stats)

In [ ]:
ax = folder_percent.plot(kind='bar', stacked=True, figsize=(10,6))
plt.title('Movement (IsMoving) by Folder')
plt.ylabel('Percentage (%)')
plt.xlabel('Folder')
plt.legend(title='IsMoving')
plt.xticks(rotation=45)

# Annotate with counts
for container, counts in zip(ax.containers, folder_counts.T.values):
    ax.bar_label(container, labels=counts, label_type='center', fontsize=8, color="white")

plt.tight_layout()
plt.show()


# **Moving Wagon data analysis**

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Make sure StartTime is datetime
df_pjm['Date'] = pd.to_datetime(df_pjm['Date'], errors='coerce')

# Filter only rows where IsMoving = True
moving_df = df_pjm[df_pjm['IsMoving'] == True].copy()

# Extract the day
moving_df['Day'] = moving_df['Date'].dt.date
moving_df['Day'] = pd.to_datetime(moving_df['Day'])  # ensure datetime64[ns]

# Count number of moving records per Folder per Day
movement_counts = moving_df.groupby(['Folder', 'Day']).size().reset_index(name='Count')

# Plot
plt.figure(figsize=(12,6))

for idx, folder in enumerate(movement_counts['Folder'].unique()):
    folder_data = movement_counts[movement_counts['Folder'] == folder]
    plt.scatter(folder_data['Day'], [idx]*len(folder_data), 
                s=folder_data['Count']*20,   # scale size of dots
                alpha=0.6, 
                label=folder)

plt.title('Days with Movement per Folder (Dot Size = Movement Counts)')
plt.xlabel('Date')
plt.ylabel('Folder')
plt.yticks(range(len(movement_counts['Folder'].unique())), movement_counts['Folder'].unique())

# Set ticks every 2 days
plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=2))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

plt.xticks(rotation=45)
plt.legend(title="Folder", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, axis='x')
plt.tight_layout()
plt.show()


In [ ]:
# --- 1. Calculate stats ---
invalid_percent = moving_df.groupby('Folder')['InvalidDateFlag'].mean() * 100
invalid_counts_full = moving_df.groupby('Folder')['InvalidDateFlag'].sum()
total_counts_full = moving_df.groupby('Folder').size()

# --- 2. Plot ---
fig, ax = plt.subplots(figsize=(10,5))

# Background: total counts
ax.bar(total_counts_full.index, total_counts_full.values, 
       color='skyblue', alpha=0.5, label='Total Records')

# Foreground: invalid counts
ax.bar(invalid_counts_full.index, invalid_counts_full.values, 
       color='tomato', alpha=0.8, label='Invalid Records')

# Annotate % above red bars
for i, (count, perc) in enumerate(zip(invalid_counts_full, invalid_percent)):
    ax.text(i, count + 1, f"{perc:.1f}%", 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Labels & style
ax.set_title('InvalidDateFlag Records per Folder')
ax.set_xlabel('Folder')
ax.set_ylabel('Record Count')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
ax.legend()

plt.tight_layout()
plt.show()


# --- 3. Pie chart (share of invalids among all invalids) ---

# Calculate invalid counts
invalid_counts = df_pjm[df_pjm['InvalidDateFlag']==1]['Folder'].value_counts()

# Explode the largest slice
explode = [0.05 if i == invalid_counts.idxmax() else 0 for i in invalid_counts.index]

plt.figure(figsize=(8,8))
wedges, texts, autotexts = plt.pie(
    invalid_counts,
    autopct='%1.1f%%', 
    startangle=90,
    explode=explode,
    colors=plt.cm.tab20.colors,
    textprops={'fontsize':10}  # smaller text inside pie
)

# Add legend with folder names and counts
plt.legend(
    wedges,
    [f"{folder} ({count})" for folder, count in invalid_counts.items()],
    title="Folder",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
    fontsize=9
)

plt.title('Share of Invalid Records by Folder', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
moving_df.head()

In [ ]:
import pandas as pd
import numpy as np

mdf = moving_df.copy()

# Coerce dtypes
mdf['StartTime'] = pd.to_datetime(mdf['StartTime'], errors='coerce')
mdf['EndTime']   = pd.to_datetime(mdf['EndTime'],   errors='coerce')
for c in ['MsgsCount','RPM0Count','RPM308Count','InvalidGPSCount']:
    mdf[c] = pd.to_numeric(mdf[c], errors='coerce').fillna(0)

# Valid times & duration
valid_time = mdf['StartTime'].notna() & mdf['EndTime'].notna()
duration_min = (mdf['EndTime'] - mdf['StartTime']).dt.total_seconds().div(60)

# Safe denominator for ratios
msgs = mdf['MsgsCount'].where(mdf['MsgsCount'] > 0, 1)
# Default: invalid (=1)
mdf['InvalidDuration'] = 1

# Set to 0 when rule passes (valid_time & duration >= 40)
mdf.loc[valid_time & (duration_min >= 40), 'InvalidDuration'] = 0
# Flags
# Ensure integer dtype
mdf['InvalidDuration'] = mdf['InvalidDuration'].astype(int)
mdf['LowMsgs']         = mdf['MsgsCount'] < 2400
mdf['HighRPM0']        = (mdf['RPM0Count']   / msgs) > 0.1
mdf['HighRPM308']      = (mdf['RPM308Count'] / msgs) > 0.1
mdf['HighGPS0']        = (mdf['InvalidGPSCount'] / msgs) > 0.1

# to 0/1
flag_cols = ['InvalidDuration','LowMsgs','HighRPM0','HighRPM308','HighGPS0']
mdf[flag_cols] = mdf[flag_cols].astype(int)

moving_df_binary = mdf
moving_df_binary.isna().sum()


In [ ]:
import matplotlib.pyplot as plt

# Count total rows per Folder, sorted alphabetically
folder_totals = moving_df_binary.groupby('Folder').size().sort_index()

# Plot bar chart
plt.figure(figsize=(10,6))
bars = folder_totals.plot(kind='bar', color='skyblue', alpha=0.8)

plt.title('Total Number of Files per Folder')
plt.xlabel('Folder')
plt.ylabel('Number of Files')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate with counts
for i, v in enumerate(folder_totals):
    plt.text(i, v + 0.5, str(int(v)), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Count total messages per Folder, sorted alphabetically
messages_totals = moving_df_binary.groupby('Folder')['MsgsCount'].mean().sort_index()

# Plot bar chart
plt.figure(figsize=(10,6))
bars = messages_totals.plot(kind='bar', color='skyblue', alpha=0.8)

plt.title('Average Number of Messages per Folder')
plt.xlabel('Folder')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate with counts
for i, v in enumerate(messages_totals):
    plt.text(i, v + 0.5, str(int(v)), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

moving_df_binary['AcqPeriod_sec'] = pd.to_timedelta(moving_df_binary['AcqPeriod']).dt.total_seconds()
# Count avg acquisition time per Folder, sorted alphabetically
messages_totals = moving_df_binary.groupby('Folder')['AcqPeriod_sec'].mean().sort_index()
messages_totals = messages_totals.fillna(0)

# Plot bar chart
plt.figure(figsize=(10,6))
bars = messages_totals.plot(kind='bar', color='skyblue', alpha=0.8)

plt.title('Average Acquisition period per Folder')
plt.xlabel('Folder')
plt.ylabel('Time (seconds)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate with counts
for i, v in enumerate(messages_totals):
    plt.text(i, v + 0.5, str(int(v)), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Count total InvalidGPSCount per Folder
gps_invalid_counts = moving_df_binary.groupby('Folder')['InvalidGPSCount'].sum().sort_index()

print("InvalidGPSCount per Folder:")
print(gps_invalid_counts)

# Optional: Bar chart
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
gps_invalid_counts.plot(kind='bar', color='orange', alpha=0.8)

plt.title('Total Invalid GPS Messages per Folder')
plt.xlabel('Folder')
plt.ylabel('Invalid GPS Messages Count')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Annotate bars
for i, v in enumerate(gps_invalid_counts):
    plt.text(i, v + 0.5, str(int(v)), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure StartTime is datetime
moving_df_binary['StartTime'] = pd.to_datetime(moving_df_binary['StartTime'])

# Extract date only
moving_df_binary['Date'] = moving_df_binary['StartTime'].dt.date

# Group by Folder and Date, get median NominalFreq
mean_freq = (
    moving_df_binary.groupby(['Folder', 'Date'])['NominalFreq']
    .mean()
    .reset_index()
)

# --- Plot ---
plt.figure(figsize=(12,6))
for folder, group in mean_freq.groupby('Folder'):
    plt.scatter(group['Date'], group['NominalFreq'], marker='o', label=folder)

plt.title('Average NominalFreq per Folder')
plt.xlabel('Date')
plt.ylabel('NominalFreq (Average)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.7)
plt.legend(title='Folder')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df = moving_df_binary.copy()

# Ensure types
df['StartTime'] = pd.to_datetime(df['StartTime'], errors='coerce')
df['Date'] = df['StartTime'].dt.date

# One row per file: average NominalFreq for that file (if duplicates exist)
per_file = (
    df.groupby(['Folder','Filename'], as_index=False)
      .agg(NominalFreq=('NominalFreq','mean'),
           Date=('Date','min'),                 # first date for the file
           MsgsCount=('MsgsCount','sum'))       # optional for sizing
)

# Drop files without a frequency
per_file = per_file.dropna(subset=['NominalFreq'])

# --- Plot: each dot = one file's avg sampling freq ---
plt.figure(figsize=(12,6))
for folder, g in per_file.groupby('Folder'):
    plt.scatter(g['Date'], g['NominalFreq'], marker='o', label=folder, alpha=0.8)

plt.title('Average Sampling Frequency per File')
plt.xlabel('Date')
plt.ylabel('NominalFreq (Hz)')  # adjust unit label if different
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

plt.xticks(rotation=45)
plt.grid(True, alpha=0.7)
plt.legend(title='Folder')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# List of problem columns
problem_cols = ['InvalidDuration', 'LowMsgs', 'HighRPM0', 'HighRPM308', 'HighGPS0']

# Total rows per Folder (sorted for consistent order)
folder_totals = moving_df_binary.groupby('Folder').size().sort_index()

# Loop through each problem
for col in problem_cols:
    # Problem counts per Folder (match ordering)
    problem_counts = moving_df_binary.groupby('Folder')[col].sum().reindex(folder_totals.index)
    problem_percent = (problem_counts / folder_totals * 100).round(1)

    # Create figure
    fig, ax1 = plt.subplots(figsize=(10,6))

    # Plot total folder counts (background)
    ax1.bar(folder_totals.index, folder_totals.values, color='skyblue', alpha=0.5, label='Total Records')

    # Plot problem counts (overlay)
    ax1.bar(folder_totals.index, problem_counts.values, color='tomato', alpha=0.8, label=f'{col} Cases')

    # Limits and labels
    # ax1.set_ylim(0, 100)
    ax1.set_title(f'{col} by Folder')
    ax1.set_xlabel('Folder')
    ax1.set_ylabel('Count')
    ax1.grid(axis='y', linestyle='--', alpha=0.7)

    # Annotate problem bars with count and %
    for i, (count, perc) in enumerate(zip(problem_counts, problem_percent)):
        ax1.text(i, count + 1, f"{int(count)} ({perc}%)", 
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Legend and final layout
    ax1.legend(loc='upper right')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


**This following plot group all the acquisition of one single day summing all the messages count within those days**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate by Folder and Day
agg_counts = moving_df.groupby(['Folder', moving_df['Date'].dt.date])[
    ['MsgsCount', 'InvalidGPSCount', 'RPM0Count', 'RPM308Count']
].sum().reset_index().rename(columns={'Date': 'Day'})

# Melt for plotting
agg_melted = agg_counts.melt(id_vars=['Folder', 'Day'], 
                             value_vars=['MsgsCount', 'InvalidGPSCount', 'RPM0Count', 'RPM308Count'],
                             var_name='Metric', value_name='Count')

# Use a more distinct color palette
palette = sns.color_palette("Set2", 4)

# Create separate figures for each Folder
for folder in agg_melted['Folder'].unique():
    folder_data = agg_melted[agg_melted['Folder'] == folder]
    
    plt.figure(figsize=(10,6))
    sns.barplot(data=folder_data, x="Day", y="Count", hue="Metric", palette=palette)
    plt.title(f"Sensor Data Acquisition Metrics per Day (Moving) - Folder: {folder}")
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.legend(title="Metric")
    plt.tight_layout()
    plt.show()


# **Daily Metrics of Moving Wagon**

In [ ]:
# --- 3. Speed & Distance During Movement ---
plt.figure(figsize=(12,5))

# Histogram for AvgSpeed
df_pjm[df_pjm['IsMoving']]['AvgSpeed'].hist(bins=30, alpha=0.7, label='AvgSpeed')
df_pjm[df_pjm['IsMoving']]['MaxSpeed'].hist(bins=30, alpha=0.4, label='MaxSpeed', color='orange')
plt.title('Distribution of Speed During Movement')
plt.xlabel('Speed (km/h)')
plt.ylabel('Count')
plt.legend()

In [ ]:
import matplotlib.pyplot as plt

# Filter only rows where IsMoving == True
moving_df = df_pjm[df_pjm['IsMoving'] == True]

plt.figure(figsize=(12,6))

# Plot StartVbatt
plt.subplot(1,2,1)
moving_df.boxplot(column='StartVbatt')
plt.title('Start Battery Voltage (When Moving)')
plt.ylabel('Voltage (V)')
plt.xlabel('')

# Plot EndVbatt
plt.subplot(1,2,2)
moving_df.boxplot(column='EndVbatt')
plt.title('End Battery Voltage (When Moving)')
plt.ylabel('Voltage (V)')
plt.xlabel('')

plt.tight_layout()
plt.show()


# **Pressure Data Availability**

In [ ]:
print(df_p.shape)
df_p.head()

**Here we want to search for the availability of pressure data within the time range of each pjm file counted from "Date" Column + 1 hour**

In [ ]:
# Filter data where IsMoving == True
df_p_moving = df_p[df_p['IsMoving'] == True]

# Count True/False in hasPressure
pressure_counts = df_p_moving['HasPressure'].value_counts()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(
    pressure_counts,
    labels=pressure_counts.index,
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Distribution of Pressure Data Availability')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1) Build counts & percentages
folder_pressure = (
    df_p_moving.groupby('Folder')['HasPressure']
    .value_counts().unstack(fill_value=0)
)
folder_pressure_pct = (
    df_p_moving.groupby('Folder')['HasPressure']
    .value_counts(normalize=True).unstack(fill_value=0) * 100.0
)

# 2) Normalize columns to {0,1} no matter if they are booleans/strings/ints
def to01(col):
    if col in (True, 'True', 'true', 1, '1'):
        return 1
    if col in (False, 'False', 'false', 0, '0'):
        return 0
    return col  # leave others as-is

folder_pressure = folder_pressure.rename(columns={c: to01(c) for c in folder_pressure.columns})
folder_pressure_pct = folder_pressure_pct.rename(columns={c: to01(c) for c in folder_pressure_pct.columns})

# 3) Ensure both 0 and 1 columns exist and order them
for c in [0, 1]:
    if c not in folder_pressure.columns:
        folder_pressure[c] = 0
    if c not in folder_pressure_pct.columns:
        folder_pressure_pct[c] = 0.0

folder_pressure = folder_pressure[[0, 1]]
folder_pressure_pct = folder_pressure_pct[[0, 1]]

# 4) Plot stacked bar (COUNTS) and annotate only True (%) inside the True segment
ax = folder_pressure.plot(kind='bar', stacked=True, figsize=(10, 6), color=['lightgray', 'steelblue'])
plt.title('HasPressure by Folder (Moving Only)')
plt.ylabel('Count')
plt.xlabel('Folder')
plt.legend(title='HasPressure', labels=['False (0)', 'True (1)'])
plt.xticks(rotation=45)

for i, folder in enumerate(folder_pressure.index):
    true_count = folder_pressure.loc[folder, 1]
    if true_count > 0:
        false_count = folder_pressure.loc[folder, 0]
        pct_true = folder_pressure_pct.loc[folder, 1]
        y_pos = false_count + true_count / 2.0
        ax.text(i, y_pos, f"{pct_true:.1f}%",
                ha='center', va='center', color='white', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
df_p_moving['Day'] = df_p_moving['Date'].dt.date

# Calculate daily percentage of HasPressure per Folder
pressure_trend = (
    df_p_moving.groupby(['Folder', 'Day'])['HasPressure']
    .mean()
    .reset_index()
)

# Convert to percentage
pressure_trend['HasPressurePct'] = pressure_trend['HasPressure'] * 100

# Plot trend per Folder
plt.figure(figsize=(14, 6))
for folder in pressure_trend['Folder'].unique():
    folder_data = pressure_trend[pressure_trend['Folder'] == folder]
    plt.scatter(
        folder_data['Day'],
        folder_data['HasPressurePct'],
        marker='o',
        label=folder,
    )

plt.title('Trend of Pressure Data Availability Over Time (Moving Only)')
plt.xlabel('Date')
plt.ylabel('HasPressure (%)')
plt.xticks(rotation=45)
plt.legend(title='Folder')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.lines as mlines

# --- Ensure Day exists (derive from StartTime or Date) ---
if 'Day' not in df_p_moving.columns:
    if 'StartTime' in df_p_moving.columns:
        df_p_moving['Day'] = pd.to_datetime(df_p_moving['StartTime'], errors='coerce').dt.date
    elif 'Date' in df_p_moving.columns:
        df_p_moving['Day'] = pd.to_datetime(df_p_moving['Date'], errors='coerce').dt.date
    else:
        raise KeyError("Need a 'Day' (or StartTime/Date) column to compute daily percentages.")

# --- Clean HasPressure to numeric 0/1 (treat NaN as 0) ---
df_p_moving['HasPressure'] = (
    pd.to_numeric(df_p_moving['HasPressure'], errors='coerce')
    .fillna(0.0)
    .clip(0, 1)
)

# --- Compute per (Folder, Day) percentage and write it to every row in that group ---
df_p_moving['HasPressurePct'] = (
    df_p_moving
    .groupby(['Folder', 'Day'], dropna=False)['HasPressure']
    .transform('mean') * 100.0
)
# Scale bubble sizes
files = df_p_moving['NumPressureFiles'].to_numpy()
fmin, fmax = files.min(), files.max()
sizes = 50 + (files - fmin) * (200 - 50) / (fmax - fmin + 1e-9)
df_p_moving['BubbleSize'] = sizes

plt.figure(figsize=(14, 6))

for folder, g in df_p_moving.groupby('Folder'):
    plt.scatter(
        g['Day'], g['HasPressurePct'],
        s=g['BubbleSize'],
        alpha=0.7,
        edgecolors='k',
        linewidths=0.5,
        label=None   # don't auto-add to legend
    )

# Custom legend handles with fixed size
handles = [mlines.Line2D([], [], color='steelblue', marker='o', linestyle='',
                         markersize=6, label=folder)
           for folder in df_p_moving['Folder'].unique()]

plt.legend(handles=handles, title='Folder', bbox_to_anchor=(1.02, 1), loc='upper left')

# Tick every 2 days
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---- 1) Normalize all x data to tz-naive midnight datetimes ----
def to_day(x):
    # Works for strings, dates, datetimes (tz-aware or not)
    s = pd.to_datetime(x, errors='coerce', utc=True)      # parse with UTC
    s = s.dt.tz_convert('UTC').dt.tz_localize(None)       # make tz-naive
    return s.dt.normalize()                                # midnight

# Pressure df
dfp = df_p_moving.copy()
if 'Day' in dfp:
    dfp['Day'] = to_day(dfp['Day'])
elif 'StartTime' in dfp:
    dfp['Day'] = to_day(dfp['StartTime'])
elif 'Date' in dfp:
    dfp['Day'] = to_day(dfp['Date'])
else:
    raise KeyError("Need Day/StartTime/Date in df_p_moving")

# Movement df
dfj = df_pjm.copy()
dfj['Day'] = to_day(dfj.get('Date', pd.NaT))
moving_df = dfj[dfj['IsMoving'] == True].copy()

# Safety: drop rows with missing Day for plotting
dfp = dfp.dropna(subset=['Day'])
moving_df = moving_df.dropna(subset=['Day'])

# ---- 2) Recompute movement counts and pressure % (if needed) ----
movement_counts = (
    moving_df.groupby(['Folder','Day']).size()
    .reset_index(name='Count')
    .sort_values(['Day','Folder'])
)

# (Only if you need HasPressurePct)
dfp['HasPressure'] = pd.to_numeric(dfp['HasPressure'], errors='coerce').fillna(0).clip(0,1)
dfp['HasPressurePct'] = (
    dfp.groupby(['Folder','Day'])['HasPressure'].transform('mean') * 100.0
)

# ---- 3) Shared x-limits & ticks ----
xmin = min(dfp['Day'].min(), movement_counts['Day'].min())
xmax = max(dfp['Day'].max(), movement_counts['Day'].max())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios':[2,1]})

# Top plot (pressure)
for folder, g in dfp.groupby('Folder'):
    ax1.scatter(g['Day'], g['HasPressurePct'], s=80, alpha=0.7, edgecolors='k', linewidths=0.4, label=folder)
ax1.set_title('Pressure Data Availability Over Time')
ax1.set_ylabel('HasPressure (%)')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(
    title='Folder',
    loc='upper right',   # inside top-right
    frameon=True,        # box around legend (optional)
    fontsize=9,
    title_fontsize=10,
    scatterpoints=1,
    markerscale=0.8
)
# Bottom plot (movement)
folders = sorted(movement_counts['Folder'].unique())
ypos = {f:i for i,f in enumerate(folders)}
for folder, g in movement_counts.groupby('Folder'):
    ax2.scatter(g['Day'], [ypos[folder]]*len(g), s=50 + 20*g['Count'], alpha=0.7, edgecolors='k', linewidths=0.4, label=None)
ax2.set_yticks(list(ypos.values()))
ax2.set_yticklabels(list(ypos.keys()))
ax2.set_title('Wagon Movement Timetable (dot size = movement count)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Folder')
ax2.grid(True, axis='x', linestyle='--', alpha=0.6)

# X formatting (identical on both)
for ax in (ax1, ax2):
    ax.set_xlim(xmin.normalize(), xmax.normalize())
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.margins(x=0.01)

plt.setp(ax2.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---- 1) Normalize all x data to tz-naive midnight datetimes ----
def to_day(x):
    s = pd.to_datetime(x, errors='coerce', utc=True)      # parse with UTC
    s = s.dt.tz_convert('UTC').dt.tz_localize(None)       # make tz-naive
    return s.dt.normalize()                                # midnight

# Pressure df
dfp = df_p_moving.copy()
if 'Day' in dfp:
    dfp['Day'] = to_day(dfp['Day'])
elif 'StartTime' in dfp:
    dfp['Day'] = to_day(dfp['StartTime'])
elif 'Date' in dfp:
    dfp['Day'] = to_day(dfp['Date'])
else:
    raise KeyError("Need Day/StartTime/Date in df_p_moving")

# Movement df
dfj = df_pjm.copy()
dfj['Day'] = to_day(dfj.get('Date', pd.NaT))
moving_df = dfj[dfj['IsMoving'] == True].copy()

# Safety: drop rows with missing Day for plotting
dfp = dfp.dropna(subset=['Day'])
moving_df = moving_df.dropna(subset=['Day'])

# ✅ ---- NEW: Filter for last 30 days ----
today = pd.Timestamp.now().normalize()
start_date = today - pd.Timedelta(days=30)

dfp = dfp[dfp['Day'] >= start_date]
moving_df = moving_df[moving_df['Day'] >= start_date]

# ---- 2) Recompute movement counts and pressure % (if needed) ----
movement_counts = (
    moving_df.groupby(['Folder','Day']).size()
    .reset_index(name='Count')
    .sort_values(['Day','Folder'])
)

# (Only if you need HasPressurePct)
dfp['HasPressure'] = pd.to_numeric(dfp['HasPressure'], errors='coerce').fillna(0).clip(0,1)
dfp['HasPressurePct'] = (
    dfp.groupby(['Folder','Day'])['HasPressure'].transform('mean') * 100.0
)

# ---- 3) Shared x-limits & ticks ----
xmin = min(dfp['Day'].min(), movement_counts['Day'].min())
xmax = max(dfp['Day'].max(), movement_counts['Day'].max())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios':[2,1]})

# Top plot (pressure)
for folder, g in dfp.groupby('Folder'):
    ax1.scatter(g['Day'], g['HasPressurePct'], s=80, alpha=0.7, edgecolors='k', linewidths=0.4, label=folder)
ax1.set_title('Pressure Data Availability (Last 30 Days)')
ax1.set_ylabel('HasPressure (%)')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(
    title='Folder',
    loc='upper right',
    frameon=True,
    fontsize=9,
    title_fontsize=10,
    scatterpoints=1,
    markerscale=0.8
)

# Bottom plot (movement)
folders = sorted(movement_counts['Folder'].unique())
ypos = {f:i for i,f in enumerate(folders)}
for folder, g in movement_counts.groupby('Folder'):
    ax2.scatter(g['Day'], [ypos[folder]]*len(g), s=50 + 20*g['Count'], alpha=0.7, edgecolors='k', linewidths=0.4)
ax2.set_yticks(list(ypos.values()))
ax2.set_yticklabels(list(ypos.keys()))
ax2.set_title('Wagon Movement Timetable (Last 30 Days, dot size = movement count)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Folder')
ax2.grid(True, axis='x', linestyle='--', alpha=0.6)

# X formatting (identical on both)
for ax in (ax1, ax2):
    ax.set_xlim(start_date, today)
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax.margins(x=0.01)

plt.setp(ax2.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# Calendar-style matrix (30 days × folders)
# -----------------------------

# 1) Build canonical 30-day date index (left→right)
day_index = pd.date_range(start=start_date, end=today, freq='D', inclusive='left')  # 30 days

# 2) Per-day movement flag per (Folder, Day)
move_flag = (
    moving_df.groupby(['Folder','Day']).size()
    .rename('MoveCount').reset_index()
)
move_flag['Moved'] = move_flag['MoveCount'] > 0
move_flag = move_flag[['Folder','Day','Moved']]

# 3) Per-day pressure flag per (Folder, Day) with a threshold
PRESSURE_OK_THRESH = 50.0  # percent; adjust as needed
press_flag = (
    dfp.groupby(['Folder','Day'])['HasPressure'].mean().mul(100.0)
    .rename('HasPressurePct').reset_index()
)
press_flag['HasPressureOK'] = press_flag['HasPressurePct'] >= PRESSURE_OK_THRESH
press_flag = press_flag[['Folder','Day','HasPressureOK']]

# 4) Merge to categorical state
cal = pd.merge(move_flag, press_flag, on=['Folder','Day'], how='outer').fillna(False)
# Encode state: 0 none, 1 move-only, 2 press-only, 3 both
cal['State'] = (
    cal['Moved'].astype(int) + 2*cal['HasPressureOK'].astype(int)
)

# 5) Ensure full grid for all folders × day_index
folders = sorted(set(cal['Folder']))
grid = (
    pd.MultiIndex.from_product([folders, day_index], names=['Folder','Day'])
    .to_frame(index=False)
    .merge(cal[['Folder','Day','State']], on=['Folder','Day'], how='left')
    .fillna({'State':0})
)

# 6) Pivot to matrix (rows=folders, cols=days)
mat = grid.pivot(index='Folder', columns='Day', values='State').reindex(folders)

# 7) Plot as a categorical heatmap (no seaborn)
fig, ax = plt.subplots(1, 1, figsize=(min(22, 1.0*len(day_index)+6), 0.45*len(folders)+3))

# Define a discrete colormap (0 none, 1 move, 2 pressure, 3 both)
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap([
    '#f0f0f0',  # 0: none
    '#6baed6',  # 1: moved only (blue-ish)
    '#74c476',  # 2: pressure only (green-ish)
    '#fd8d3c',  # 3: both (orange-ish)
])
norm = BoundaryNorm([ -0.5, 0.5, 1.5, 2.5, 3.5 ], cmap.N)

im = ax.imshow(mat.values, aspect='auto', cmap=cmap, norm=norm)

# Axes ticks & labels
ax.set_yticks(range(len(folders)))
ax.set_yticklabels(folders)
# Show fewer x ticks for readability
xticks = range(0, len(day_index), max(1, len(day_index)//12))
ax.set_xticks(xticks)
ax.set_xticklabels([d.strftime('%Y-%m-%d') for d in day_index[xticks]], rotation=45, ha='right')

ax.set_title(f'Wagon activity vs pressure (last 30 days, pressure≥{PRESSURE_OK_THRESH:.0f}%)')
ax.set_xlabel('Date')
ax.set_ylabel('Folder')

# Gridlines between cells (optional)
ax.set_xticks(np.arange(-.5, mat.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, mat.shape[0], 1), minor=True)
ax.grid(which='minor', color='white', linewidth=0.5)
ax.tick_params(which='minor', bottom=False, left=False)

# Legend (custom patches)
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='#f0f0f0', label='No movement & no pressure'),
    mpatches.Patch(color='#6baed6', label='Movement only'),
    mpatches.Patch(color='#74c476', label='Pressure only'),
    mpatches.Patch(color='#fd8d3c', label='Movement + Pressure'),
]
ax.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0.)

plt.tight_layout()
plt.show()

# -----------------------------
# Optional: per-wagon summary table
# -----------------------------
summary = (
    grid.assign(
        MoveOnly = grid['State'].eq(1),
        PressOnly = grid['State'].eq(2),
        Both = grid['State'].eq(3)
    )
    .groupby('Folder')[['MoveOnly','PressOnly','Both']]
    .sum()
    .assign(
        DaysAnyMove = lambda d: d['MoveOnly'] + d['Both'],
        DaysAnyPress = lambda d: d['PressOnly'] + d['Both'],
        DaysBoth = lambda d: d['Both'],
        CoveragePct = lambda d: 100.0 * d['DaysBoth'] / len(day_index)
    )
    .sort_values(['DaysBoth','DaysAnyMove','DaysAnyPress'], ascending=False)
)
print(summary)
